In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('creditcard.csv')

# First thing — understand the imbalance
print(df['Class'].value_counts())
print(f"Fraud: {df['Class'].mean()*100:.3f}%")

# Plot 1: Class distribution
sns.countplot(x='Class', data=df)
plt.title('Class distribution (0=Normal, 1=Fraud)')
plt.savefig('class_dist.png')

# Plot 2: Transaction amount by class
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df[df['Class']==0]['Amount'].hist(ax=axes[0], bins=50)
axes[0].set_title('Normal transaction amounts')
df[df['Class']==1]['Amount'].hist(ax=axes[1], bins=50, color='red')
axes[1].set_title('Fraud transaction amounts')
plt.savefig('amount_dist.png')

# Plot 3: Correlation heatmap (top features)
corr = df.corr()['Class'].drop('Class').abs().sort_values(ascending=False)
print("Top correlated features:\n", corr.head(10))

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# Scale Amount and Time (V1-V28 are already PCA-scaled)
scaler = StandardScaler()
df['Amount_scaled'] = scaler.fit_transform(df[['Amount']])
df['Time_scaled']   = scaler.fit_transform(df[['Time']])
df.drop(['Amount', 'Time'], axis=1, inplace=True)

X = df.drop('Class', axis=1)
y = df['Class']

# Stratified split preserves the 0.17% ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Handle imbalance: SMOTE creates synthetic fraud samples
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

print(f"Before SMOTE: {y_train.value_counts().to_dict()}")
print(f"After SMOTE:  {y_train_res.value_counts().to_dict()}")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost':             XGBClassifier(n_estimators=100, random_state=42, 
                                          scale_pos_weight=len(y_train[y_train==0])/len(y_train[y_train==1]))
}

results = {}
for name, model in models.items():
    model.fit(X_train_res, y_train_res)
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    results[name] = {
        'model':  model,
        'y_pred': y_pred,
        'auc':    roc_auc_score(y_test, y_proba)
    }
    print(f"\n{'='*40}")
    print(f"{name}")
    print(classification_report(y_test, y_pred, target_names=['Normal','Fraud']))
    print(f"AUC-ROC: {results[name]['auc']:.4f}")

In [ ]:
from sklearn.metrics import (confusion_matrix, roc_curve, 
                              precision_recall_curve, ConfusionMatrixDisplay)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (name, res) in enumerate(results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    ConfusionMatrixDisplay(cm, display_labels=['Normal','Fraud']).plot(ax=axes[i])
    axes[i].set_title(f'{name}\nAUC: {res["auc"]:.4f}')

plt.tight_layout()
plt.savefig('confusion_matrices.png')

# AUC-ROC curves — all three on one plot
plt.figure(figsize=(8, 6))
for name, res in results.items():
    y_proba = res['model'].predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC={res['auc']:.3f})")

plt.plot([0,1],[0,1],'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('AUC-ROC Curve Comparison')
plt.legend()
plt.savefig('roc_curves.png')